In [ ]:
import copernicusmarine

ds = copernicusmarine.subset(
    dataset_id="cmems_mod_glo_phy_my_0.083deg_P1D-m",
    minimum_longitude=-742.400,
    maximum_longitude=-742.410,
    minimum_latitude=-452.000,
    maximum_latitude=-452.100,
    variables=["thetao", "so", "uo", "vo"],
    start_datetime="2020-01-01",
    end_datetime="2025-12-31",
    output_directory="coleccion",
    output_filename="copernicus_subsetCentro_110745_2020_2025.nc"
)

In [ ]:
import copernicusmarine
import time
#110557
# 1. Ampliamos el Buffer para escapar de la "tierra firme"
lat_centro = -45.3214
lon_centro = -73.3567
buffer = 0.15 # ¡Cambio clave! Ahora buscaremos en un radio más amplio

min_lon = lon_centro - buffer
max_lon = lon_centro + buffer
min_lat = lat_centro - buffer
max_lat = lat_centro + buffer

directorio_hdd = "coleccion/Historico_110557"

print("Iniciando extracción con Buffer ampliado (0.15)...")

for anio in range(2020, 2026):
    fecha_inicio = f"{anio}-01-01"
    fecha_fin = "2025-05-01" if anio == 2025 else f"{anio}-12-31"
        
    nombre_archivo = f"110745_Fisica_{anio}_Micro.nc"
    
    print(f"\n-> Extrayendo año {anio}...")
    
    try:
        copernicusmarine.subset(
            dataset_id="cmems_mod_glo_phy_my_0.083deg_P1D-m",
            variables=["thetao", "so", "uo", "vo", "zos", "mlotst"],  # Añadí mlotst y zos para tener más variables físicas
            start_datetime=fecha_inicio,
            end_datetime=fecha_fin,
            minimum_longitude=min_lon,
            maximum_longitude=max_lon,
            minimum_latitude=min_lat,
            maximum_latitude=max_lat,
            minimum_depth=0.49, 
            maximum_depth=0.51,
            output_directory=directorio_hdd,
            output_filename=nombre_archivo,
            force_download=True
        )
        print(f"[OK] Año {anio} guardado.")
        time.sleep(3) 
        
    except Exception as e:
        print(f"[ERROR] Fallo en el año {anio}: {e}")

# Chlorophyll and Dissolved Oxygen from biogeochemistry dataset
print("Extrayendo Clorofila y Oxígeno...")

for anio in range(2020, 2026):
    fecha_inicio = f"{anio}-01-01"
    fecha_fin = "2025-05-01" if anio == 2025 else f"{anio}-12-31"
    
    nombre_archivo = f"110745_BGC_{anio}_Micro_Bioquimica.nc"
    
    print(f"\n-> Extrayendo BGC año {anio}...")
    
    try:
        copernicusmarine.subset(
            dataset_id="cmems_mod_glo_bgc_my_0.25deg_P1D-m",
            variables=["chl", "o2"],  # Chlorophyll and dissolved oxygen
            start_datetime=fecha_inicio,
            end_datetime=fecha_fin,
            minimum_longitude=min_lon,
            maximum_longitude=max_lon,
            minimum_latitude=min_lat,
            maximum_latitude=max_lat,
            minimum_depth=0.49,
            maximum_depth=0.51,
            output_directory=directorio_hdd,
            output_filename=nombre_archivo,
            force_download=True
        )
        print(f"[OK] BGC año {anio} guardado.")
        time.sleep(3)
        
    except Exception as e:
        print(f"[ERROR] Fallo en BGC año {anio}: {e}")

In [ ]:
import xarray as xr
import pandas as pd
import os

# Configuración de rutas
directorio_hdd = "coleccion/Historico_110557"
lista_dataframes_anuales = []

print("Iniciando procesamiento de matrices espaciales hacia formato tabular...")

for anio in range(2020, 2026):
    archivo_fisica = os.path.join(directorio_hdd, f"110745_Fisica_{anio}_Micro.nc")
    archivo_bgc = os.path.join(directorio_hdd, f"110745_BGC_{anio}_Micro_Bioquimica.nc")
    
    if os.path.exists(archivo_fisica) and os.path.exists(archivo_bgc):
        print(f"-> Extrayendo tensores del año {anio}...")
        
        ds_fis = xr.open_dataset(archivo_fisica)
        ds_bgc = xr.open_dataset(archivo_bgc)
        
        # Reducción espacial: colapso de las dimensiones lat/lon/depth mediante promedio
        # xarray discrimina automáticamente las variables 2D (zos, mlotst) y las promedia solo en 2 ejes.
        dim_fis_drop = [d for d in ['latitude', 'longitude', 'depth'] if d in ds_fis.dims]
        dim_bgc_drop = [d for d in ['latitude', 'longitude', 'depth'] if d in ds_bgc.dims]
        
        df_fis = ds_fis.mean(dim=dim_fis_drop).to_dataframe().reset_index()
        df_bgc = ds_bgc.mean(dim=dim_bgc_drop).to_dataframe().reset_index()
        
        # Actualización de matriz de extracción
        columnas_fis_necesarias = ['time', 'thetao', 'so', 'uo', 'vo', 'zos', 'mlotst']
        columnas_bgc_necesarias = ['time', 'chl', 'o2']
        
        df_fis = df_fis[[c for c in columnas_fis_necesarias if c in df_fis.columns]]
        df_bgc = df_bgc[[c for c in columnas_bgc_necesarias if c in df_bgc.columns]]
        
        # Operación INNER JOIN temporal
        df_anio = pd.merge(df_fis, df_bgc, on='time', how='inner')
        lista_dataframes_anuales.append(df_anio)
        
        ds_fis.close()
        ds_bgc.close()
        
    else:
        print(f"[ADVERTENCIA] Archivos incompletos para el año {anio}. Omitiendo integración.")

# Consolidación final y formateo de features
if lista_dataframes_anuales:
    df_final = pd.concat(lista_dataframes_anuales, ignore_index=True)
    
    # Asignación de nomenclatura estándar para el modelo predictivo
    df_final.rename(columns={
        'time': 'Fecha',
        'thetao': 'Temperatura_C',
        'so': 'Salinidad_psu',
        'uo': 'Corriente_E_m_s',
        'vo': 'Corriente_N_m_s',
        'zos': 'Altura_Mar_m',
        'mlotst': 'Capa_Mezcla_m',
        'chl': 'Clorofila_mg_m3',
        'o2': 'Oxigeno_mmol_m3'
    }, inplace=True)
    
    df_final.sort_values('Fecha', inplace=True)
    df_final.dropna(subset=['Fecha'], inplace=True)
    
    ruta_csv = os.path.join(directorio_hdd, "Consolidado_Ambiental_Centro_110557.csv")
    df_final.to_csv(ruta_csv, index=False, encoding='utf-8')
    
    print(f"\nPipeline ETL completado. Matriz de covariables guardada en: {ruta_csv}")
    print(df_final.head())
else:
    print("\nError fatal: No existen tensores NetCDF en el directorio especificado.")

In [ ]:
import copernicusmarine
import time

# 1. Ampliamos el Buffer para escapar de la "tierra firme"
lat_centro = -45.2050
lon_centro = -74.2406
buffer = 0.15 # ¡Cambio clave! Ahora buscaremos en un radio más amplio

min_lon = lon_centro - buffer
max_lon = lon_centro + buffer
min_lat = lat_centro - buffer
max_lat = lat_centro + buffer

directorio_hdd = "coleccion/Historico_110745"

print("Iniciando extracción con Buffer ampliado (0.15)...")

for anio in range(2020, 2026):
    fecha_inicio = f"{anio}-01-01"
    fecha_fin = "2025-05-01" if anio == 2025 else f"{anio}-12-31"
        
    nombre_archivo = f"110745_Fisica_{anio}_Micro.nc"
    
    print(f"\n-> Extrayendo año {anio}...")
    
    try:
        copernicusmarine.subset(
            dataset_id="cmems_mod_glo_phy_my_0.083deg_P1D-m",
            variables=["thetao", "so", "uo", "vo", "zos", "mlotst"],
            start_datetime=fecha_inicio,
            end_datetime=fecha_fin,
            minimum_longitude=min_lon,
            maximum_longitude=max_lon,
            minimum_latitude=min_lat,
            maximum_latitude=max_lat,
            minimum_depth=0.49, 
            maximum_depth=0.51,
            output_directory=directorio_hdd,
            output_filename=nombre_archivo,
            force_download=True
        )
        print(f"[OK] Año {anio} guardado.")
        time.sleep(3) 
        
    except Exception as e:
        print(f"[ERROR] Fallo en el año {anio}: {e}")

In [ ]:
import xarray as xr
import pandas as pd
import glob
import os
import numpy as np

# 1. Definir la ruta de origen y destino
carpeta_origen = "coleccion/Historico_110557"
archivo_salida = "coleccion/Historico_110557_Consolidado_2020_2025_2.0.csv"

# Usamos glob para encontrar todos los archivos .nc en la carpeta
# Esto hace que tu script sea modular; si mañana añades el 2026, el script lo tomará automáticamente
archivos_nc = glob.glob(os.path.join(carpeta_origen, "*.nc"))

print(f"Se encontraron {len(archivos_nc)} archivos NetCDF para procesar.")

lista_dataframes = []

# 2. Bucle de Transformación (El motor del ETL)
for archivo in archivos_nc:
    nombre_base = os.path.basename(archivo)
    print(f"Procesando: {nombre_base}...")
    
    try:
        # A) Abrir el cubo
        ds = xr.open_dataset(archivo)
        df = ds.to_dataframe().reset_index()
        
        # B) Limpieza Espacial (Eliminar la "Tierra Firme")
        # El buffer ampliado capturó agua y tierra. Aquí matamos los píxeles de tierra (NaN)
        df_agua = df.dropna(subset=['thetao', 'so', 'uo', 'vo'], how='all')
        
        # C) Filtro de Profundidad (Solo la primera capa)
        profundidad_superficie = df_agua['depth'].min()
        df_superficie = df_agua[df_agua['depth'] == profundidad_superficie]
        
        # D) Colapso Espacial (Promediar los píxeles de agua de ese día)
        df_diario = df_superficie.groupby('time').mean(numeric_only=True).reset_index()
        
        # Guardar en la lista maestra
        lista_dataframes.append(df_diario)
        
        # Cerrar el archivo para no saturar la memoria RAM
        ds.close()
        
    except Exception as e:
        print(f"[ERROR] No se pudo procesar {nombre_base}. Detalle: {e}")

print("\nUniendo todos los años históricos...")

# 3. Consolidación Final
if lista_dataframes:
    # Unir todos los años uno debajo del otro
    df_maestro = pd.concat(lista_dataframes, ignore_index=True)
    
    # Ordenar cronológicamente (por si glob leyó los archivos en desorden)
    df_maestro = df_maestro.sort_values('time')
    
    # Resetear el índice final
    df_maestro = df_maestro.reset_index(drop=True)
    
    # 4. Guardar en el disco duro
    df_maestro.to_csv(archivo_salida, index=False)
    
    print("\n¡CONSOLIDACIÓN EXITOSA!")
    print(f"Total de días históricos procesados: {len(df_maestro)}")
    print(f"Archivo guardado en: {archivo_salida}")
    
    print("\nVista previa de los primeros y últimos días:")
    print(df_maestro[['time', 'thetao', 'so']].iloc[np.r_[0:3, -3:0]]) # Muestra cabeza y cola
else:
    print("No se generó ningún DataFrame. Revisa las rutas o los archivos.")

In [ ]:
import xarray as xr
import pandas as pd

# 1. Definir las rutas (usando las que definiste en tu descarga)
ruta_nc = "coleccion/Historico_110745_2020_2025.nc"
ruta_csv = "coleccion/datos_110745_2020_2025.csv"

print("1. Abriendo el cubo de datos NetCDF...")
# Abrimos el archivo
ds = xr.open_dataset(ruta_nc)

print("2. Aplanando el cubo a una tabla de datos (DataFrame)...")
# Convertimos el cubo 4D a una tabla plana bidimensional
df = ds.to_dataframe()

# xarray crea índices múltiples (fecha, lat, lon). Los reseteamos para que sean columnas normales.
df = df.reset_index()

print("3. Limpiando datos nulos (tierra firme)...")
# Los píxeles que caen sobre tierra no tienen temperatura del agua (son NaN).
# Los eliminamos para que el CSV no pese gigabytes innecesariamente.
df_limpio = df.dropna(subset=['thetao', 'so', 'uo', 'vo', 'zos', 'mlotst', 'chl', 'o2'], how='all')

print("4. Guardando en el disco duro...")
# Exportamos a CSV
df_limpio.to_csv(ruta_csv, index=False)

print("\n¡Conversión Exitosa!")
print(f"Registros totales en el CSV: {len(df_limpio)}")
print("\nVista previa de las primeras 5 filas:")
print(df_limpio[['time', 'latitude', 'longitude', 'depth', 'thetao', 'so']].head())

# Cerramos el archivo original para liberar memoria RAM
ds.close()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# 1. Cargar tu CSV local
ruta_csv = "Historico_110745_Consolidado_2020_2025.csv"
df = pd.read_csv(ruta_csv)

# 2. Asegurar que Python entienda que 'time' es una fecha
df['time'] = pd.to_datetime(df['time'])

# 3. Configurar la "Lona" (Figura) con 4 subgráficos (filas=4, columnas=1)
fig, axs = plt.subplots(4, 1, figsize=(12, 16), sharex=True)
fig.suptitle('Análisis Oceanográfico 2020 - 2025 - Centro 110745 (Copernicus)', fontsize=16, fontweight='bold', y=0.92)

# --- Gráfico 1: Temperatura ---
axs[0].plot(df['time'], df['thetao'], color='firebrick', linewidth=1.5)
axs[0].set_ylabel('Temperatura (°C)', fontweight='bold')
axs[0].set_title('Temperatura Superficial del Mar (thetao)', loc='left')
axs[0].grid(True, linestyle='--', alpha=0.6)
# Añadimos una línea de umbral de peligro referencial para el Caligus
axs[0].axhline(y=10.5, color='red', linestyle=':', label='Umbral referencial brote')
axs[0].legend(loc='upper right')

# --- Gráfico 2: Salinidad ---
axs[1].plot(df['time'], df['so'], color='teal', linewidth=1.5)
axs[1].set_ylabel('Salinidad (PSU)', fontweight='bold')
axs[1].set_title('Salinidad Superficial (so)', loc='left')
axs[1].grid(True, linestyle='--', alpha=0.6)

# --- Gráfico 3: Corriente U (Este-Oeste) ---
axs[2].plot(df['time'], df['uo'], color='steelblue', linewidth=1)
axs[2].set_ylabel('Velocidad (m/s)', fontweight='bold')
axs[2].set_title('Corriente Zonal Este-Oeste (uo) [+ Este, - Oeste]', loc='left')
axs[2].axhline(y=0, color='black', linestyle='-', linewidth=0.8) # Línea cero
axs[2].grid(True, linestyle='--', alpha=0.6)

# --- Gráfico 4: Corriente V (Norte-Sur) ---
axs[3].plot(df['time'], df['vo'], color='purple', linewidth=1)
axs[3].set_ylabel('Velocidad (m/s)', fontweight='bold')
axs[3].set_title('Corriente Meridional Norte-Sur (vo) [+ Norte, - Sur]', loc='left')
axs[3].axhline(y=0, color='black', linestyle='-', linewidth=0.8) # Línea cero
axs[3].grid(True, linestyle='--', alpha=0.6)

# 4. Formatear el eje X (Fechas) para que se vea limpio
axs[3].xaxis.set_major_locator(mdates.MonthLocator())
axs[3].xaxis.set_major_formatter(mdates.DateFormatter('%b'))
plt.xlabel('Meses del Año 2020-2025', fontweight='bold', fontsize=12)

# Rotar las fechas un poco para que no se superpongan
plt.setp(axs[3].xaxis.get_majorticklabels(), rotation=45)

# Ajustar el espaciado y mostrar
plt.tight_layout()
plt.subplots_adjust(top=0.88)
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# 1. Cargar tu CSV local
ruta_csv = "coleccion/datos_110745_2020_2025.csv"
df = pd.read_csv(ruta_csv)

# 2. EL FILTRO DE INGENIERÍA (Arreglando el desastre dimensional)
# A) Aislar solo la capa superficial (la profundidad mínima del dataset)
profundidad_superficie = df['depth'].min()
df_superficie = df[df['depth'] == profundidad_superficie]

# B) Colapsar el mapa espacial: Promediar todos los píxeles de esa gran caja por cada día
df_limpio = df_superficie.groupby('time').mean(numeric_only=True).reset_index()

# 3. Asegurar que Python entienda la fecha
df_limpio['time'] = pd.to_datetime(df_limpio['time'])

# --- De aquí en adelante, usa df_limpio en vez de df para tus gráficos ---

fig, axs = plt.subplots(4, 1, figsize=(12, 16), sharex=True)
fig.suptitle('Análisis Oceanográfico 2020 - 2025 - Superficie', fontsize=16, fontweight='bold', y=0.92)

# Gráfico 1: Temperatura
axs[0].plot(df_limpio['time'], df_limpio['thetao'], color='firebrick', linewidth=1.5)
axs[0].set_ylabel('Temperatura (°C)', fontweight='bold')
axs[0].set_title('Temperatura Superficial del Mar (thetao)', loc='left')
axs[0].grid(True, linestyle='--', alpha=0.6)
axs[0].axhline(y=10.5, color='red', linestyle=':', label='Umbral referencial brote')
axs[0].legend(loc='upper right')

# Gráfico 2: Salinidad
axs[1].plot(df_limpio['time'], df_limpio['so'], color='teal', linewidth=1.5)
axs[1].set_ylabel('Salinidad (PSU)', fontweight='bold')
axs[1].set_title('Salinidad Superficial (so)', loc='left')
axs[1].grid(True, linestyle='--', alpha=0.6)

# Gráfico 3: Corriente U
axs[2].plot(df_limpio['time'], df_limpio['uo'], color='steelblue', linewidth=1)
axs[2].set_ylabel('Velocidad (m/s)', fontweight='bold')
axs[2].set_title('Corriente Este-Oeste (uo)', loc='left')
axs[2].axhline(y=0, color='black', linestyle='-', linewidth=0.8)
axs[2].grid(True, linestyle='--', alpha=0.6)

# Gráfico 4: Corriente V
axs[3].plot(df_limpio['time'], df_limpio['vo'], color='purple', linewidth=1)
axs[3].set_ylabel('Velocidad (m/s)', fontweight='bold')
axs[3].set_title('Corriente Norte-Sur (vo)', loc='left')
axs[3].axhline(y=0, color='black', linestyle='-', linewidth=0.8)
axs[3].grid(True, linestyle='--', alpha=0.6)

axs[3].xaxis.set_major_locator(mdates.MonthLocator())
axs[3].xaxis.set_major_formatter(mdates.DateFormatter('%b'))
plt.xlabel('Meses del Año 2020 - 2025', fontweight='bold', fontsize=12)
plt.setp(axs[3].xaxis.get_majorticklabels(), rotation=45)

plt.tight_layout()
plt.subplots_adjust(top=0.88)
plt.show()

In [ ]:
df.info()


In [ ]:
import pandas as pd

db1 =  pd.read_csv("coleccion/Historico_110557_Consolidado_2020_2025.csv")
db2 =  pd.read_csv("coleccion/Historico_110745_Consolidado_2020_2025.csv")

print(db2.describe())


